# RAG Chat with FAISS Index

This notebook implements a simple RAG (Retrieval-Augmented Generation) chat system over a FAISS vector index built from the EmergPhase_EN.pdf document.

## Setup

First, mount your Google Drive and install dependencies.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Dependencies

In [4]:
%pip install langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Load Index and Chunks

Load the FAISS vector store, embeddings model, and chunks metadata.

In [5]:
import os
import json
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Paths
DRIVE_ROOT = "/content/drive/MyDrive/iran-prosperity-rag"
INDEX_DIR = f"{DRIVE_ROOT}/data/index"
CHUNKS_JSON = f"{INDEX_DIR}/chunks.json"

# Check if paths exist
if not os.path.exists(INDEX_DIR):
    raise FileNotFoundError(f"Index directory not found: {INDEX_DIR}\nPlease run build_index.py first!")

if not os.path.exists(CHUNKS_JSON):
    raise FileNotFoundError(f"Chunks JSON not found: {CHUNKS_JSON}\nPlease run build_index.py first!")

# Load embeddings model
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Load FAISS vector store
print(f"Loading FAISS index from: {INDEX_DIR}")
vectorstore = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
print("FAISS index loaded successfully!")

# Load chunks metadata
print(f"Loading chunks metadata from: {CHUNKS_JSON}")
with open(CHUNKS_JSON, "r", encoding="utf-8") as f:
    chunks_data = json.load(f)
print(f"Loaded {len(chunks_data)} chunks from metadata")

print("\nSetup complete!")

Loading embeddings model...


/tmp/ipython-input-3108842734.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading FAISS index from: /content/drive/MyDrive/iran-prosperity-rag/data/index
FAISS index loaded successfully!
Loading chunks metadata from: /content/drive/MyDrive/iran-prosperity-rag/data/index/chunks.json
Loaded 937 chunks from metadata

Setup complete!


## Setup Retriever

Configure the retriever with top_k results.

In [6]:
top_k = 5

# Create retriever from vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})

print(f"Retriever configured with top_k={top_k}")

Retriever configured with top_k=5


## Test Retrieval

Test the retrieval system with a sample question.

In [8]:
query = "What are the main challenges discussed in the document?"

print(f"Query: {query}\n")
print("="*60)

# Retrieve documents
docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents:\n")

for rank, doc in enumerate(docs, 1):
    # Find corresponding chunk metadata
    chunk_id = None
    page = None

    # Try to find matching chunk by text content
    for chunk in chunks_data:
        if chunk["text"] == doc.page_content:
            chunk_id = chunk["chunk_id"]
            page = chunk["page"]
            break

    # Extract score if available (FAISS doesn't always return scores directly)
    score = getattr(doc, 'metadata', {}).get('score', 'N/A')

    snippet = doc.page_content[:300] + "..." if len(doc.page_content) > 300 else doc.page_content

    print(f"Rank {rank}:")
    print(f"  Score: {score}")
    print(f"  Page: {page}")
    print(f"  Chunk ID: {chunk_id}")
    print(f"  Snippet: {snippet}")
    print()

Query: What are the main challenges discussed in the document?

Retrieved 5 documents:

Rank 1:
  Score: N/A
  Page: 29
  Chunk ID: 179
  Snippet: Mitigation  measures  for  political  challenges:  
●  Inclusive  dialogue,  power-sharing,  amnesty,  transparent  communication,  inﬂuencers  
●  Retention  incentives,  clear  mandates,  training,  whistleblower  protection,  gradual  transitions  
●  Interim  legal  frameworks,  independent  ove...

Rank 2:
  Score: N/A
  Page: 28
  Chunk ID: 178
  Snippet: Anticipated  Political  Challenges:  
●  Resistance  from  factions  opposing  the  transitional  government.  
●  Potential  lack  of  cooperation  from  existing  bureaucracies.  
●  Political  instability  affecting  resource  allocation  and  decision-making  processes.   
EMERGENCY  PHASE  |   ...

Rank 3:
  Score: N/A
  Page: 132
  Chunk ID: 736
  Snippet: regulations
 
or
 
pose
 
clear
 
conﬂicts
 
of
 
interest.
 
This
 
mechanism
 
should
 
be
 
empowered
 
with
 
legal
 
au

## Answer Generation

Implement extractive answer generation with citations (no external LLM).

In [12]:
def answer_with_citations(question):
    """
    Generate an extractive answer from retrieved chunks with citations.

    Args:
        question: User's question string

    Returns:
        Answer string with citations
    """
    # Retrieve relevant documents
    docs = retriever.invoke(question)

    if not docs or len(docs) == 0:
        return "I don't know based on the provided documents."

    # Build context from retrieved chunks
    context_chunks = []
    pages_used = set()

    for doc in docs:
        # Find matching chunk metadata
        for chunk in chunks_data:
            if chunk["text"] == doc.page_content:
                context_chunks.append({
                    "text": chunk["text"],
                    "page": chunk["page"],
                    "chunk_id": chunk["chunk_id"]
                })
                if chunk["page"] is not None:
                    pages_used.add(chunk["page"])
                break

    if not context_chunks:
        return "I don't know based on the provided documents."

    import re

    def clean_text(t: str) -> str:
        t = t.replace("\n", " ")
        t = re.sub(r"\s+", " ", t)      # چند فاصله -> یکی
        t = re.sub(r"\s([.,;:!?])", r"\1", t)  # فاصله قبل علائم نگارشی
        return t.strip()



    # Extractive answer: pick 2-4 most relevant sentences from top chunks
    # Simple approach: take first 2-3 chunks and extract key sentences
    answer_sentences = []
    chunks_used = min(3, len(context_chunks))  # Use top 3 chunks

    for i in range(chunks_used):
        chunk_text = clean_text(context_chunks[i]["text"])
        sentences = chunk_text.split('. ')


        # Take first 1-2 sentences from each chunk
        sentences_to_add = min(2, len(sentences))
        for j in range(sentences_to_add):
            if sentences[j].strip():
                answer_sentences.append(sentences[j].strip() + '.')

    # Limit to 4 sentences total
    answer_sentences = answer_sentences[:4]

    if not answer_sentences:
        return "I don't know based on the provided documents."

    # Build answer
    answer = " ".join(answer_sentences)

    # Add citations
    if pages_used:
        sorted_pages = sorted(pages_used)
        citations = " ".join([f"[p.{p}]" for p in sorted_pages])
        answer += f"\n\nSources: {citations}"

    return answer

# Test the function
test_question = "What are the main challenges discussed?"
print(f"Question: {test_question}\n")
print("Answer:")
print(answer_with_citations(test_question))

Question: What are the main challenges discussed?

Answer:
Mitigation measures for political challenges: ● Inclusive dialogue, power-sharing, amnesty, transparent communication, inﬂuencers ● Retention incentives, clear mandates, training, whistleblower protection, gradual transitions ● Interim legal frameworks, independent oversight, public reporting, rapid response teams, conﬂict resolution EMERGENCY PHASE | IPP 30. Anticipated Political Challenges: ● Resistance from factions opposing the transitional government. ● Potential lack of cooperation from existing bureaucracies. transition where it matters most (e.g., rural areas that suffer from severe underdevelopment).

Sources: [p.28] [p.29] [p.123] [p.132] [p.142]


## Interactive Chat Loop

Run this cell to start an interactive chat session. Type 'exit' or press Enter with empty input to stop.

In [13]:
print("="*60)
print("RAG Chat System")
print("Ask questions about the document. Type 'exit' or press Enter to quit.")
print("="*60)
print()

while True:
    question = input("You: ").strip()

    if not question or question.lower() == 'exit':
        print("\nGoodbye!")
        break

    print("\nAnswer:")
    answer = answer_with_citations(question)
    print(answer)
    print("\n" + "-"*60 + "\n")

RAG Chat System
Ask questions about the document. Type 'exit' or press Enter to quit.

You: what this plan is about

Answer:
regime loyalists from using these assets to sabotage the transition. The plan is predicated on a scenario following the June 2025 Israeli attacks and any subsequent collapse of central authority in Iran. and initiating a democratic transition. This plan draws on crucial historical lessons, including particularly the catastrophic errors made after the 1979 revolution.

Sources: [p.3] [p.61] [p.70] [p.92]

------------------------------------------------------------



KeyboardInterrupt: Interrupted by user